# 04 — Export to LangGraph

Take a generated topology and export it as a LangGraph StateGraph config.

**Prerequisites:**
- Python 3.10+ with numpy
- Running from inside the `flybrain-connectome` checkout
- Completed [01_generate_visualize.ipynb](01_generate_visualize.ipynb)

## Cell 1 — Load a previously generated topology

In [ ]:
import sys, os, json
from pathlib import Path

HERE = Path(__file__).resolve().parents[1]
TOOLKIT = HERE / "flybrain-connectome-mcp"
for p in [TOOLKIT, TOOLKIT / "tools"]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Load the topology
org_dir = TOOLKIT / "generated_organisms"
json_path = org_dir / "demo_predictive_coding_v2_m5_a8.json"
if not json_path.exists():
    print("[warn] Run 'flybrain demo' first to generate a topology")
    import subprocess
    subprocess.run([sys.executable, str(HERE / "flybrain_cli.py"), "demo", "--no-open"],
                   cwd=str(HERE), check=True)

with open(json_path, "r", encoding="utf-8") as f:
    topo = json.load(f)

nodes = topo.get("organism", {}).get("nodes", [])
edges = topo.get("organism", {}).get("edges", [])
print(f"[loaded] {len(nodes)} nodes, {len(edges)} edges")

## Cell 2 — Export to LangGraph format

In [ ]:
# Use the export command
import subprocess

result = subprocess.run(
    [sys.executable, str(HERE / "flybrain_cli.py"), "export", str(json_path),
     "--format", "langgraph", "--output", str(json_path.parent / "langgraph_config.json")],
    capture_output=True, text=True, cwd=str(HERE)
)
print(result.stdout)
if result.stderr:
    print("[stderr]", result.stderr)

## Cell 3 — Inspect the LangGraph config

In [ ]:
with open(json_path.parent / "langgraph_config.json", "r", encoding="utf-8") as f:
    langgraph = json.load(f)

print("LangGraph config structure:")
print(f"  blueprint : {langgraph['blueprint']}")
print(f"  format    : {langgraph['format']}")
print(f"  nodes     : {len(langgraph['graph']['nodes'])}")
print(f"  edges     : {len(langgraph['graph']['edges'])}")
print(f"  entry     : {langgraph['graph']['entry_point']}")
print(f"  exit      : {langgraph['graph']['exit_point']}")
print()

# Show first 3 nodes
print("First 3 nodes:")
for n in langgraph['graph']['nodes'][:3]:
    print(f"  {n['function']:40s}  role={n['role']:15s}  module={n['module']}")

## Cell 4 — Use with LangGraph (pseudocode)

In [ ]:
# Pseudocode for LangGraph integration (requires langgraph installed)
# The config maps directly to LangGraph StateGraph.

# from langgraph.graph import StateGraph, END
# from langgraph.checkpoint.memory import MemorySaver

# # Build state schema
# class State(dict):
#     input: str
#     output: str
#     history: list

# # Create graph
# graph = StateGraph(State)

# # Add nodes (implement each as a callable)
# for node in langgraph['graph']['nodes']:
#     def make_node(node_def):
#         async def _node(state):
#             # Implement your agent logic here
#             return {"output": "result"}
#         return _node
#     graph.add_node(node['function'], make_node(node))

# # Add edges
# for edge in langgraph['graph']['edges']:
#     graph.add_edge(edge['source'], edge['target'])

# # Compile and run
# graph.set_entry_point(langgraph['graph']['entry_point'])
# graph.set_finish_point(langgraph['graph']['exit_point'])
# compiled = graph.compile(checkpointer=MemorySaver())

print("LangGraph integration ready!")
print(f"  Nodes to implement: {len(langgraph['graph']['nodes'])}")
print(f"  Edges to wire    : {len(langgraph['graph']['edges'])}")

## Key insight

LangGraph is great for stateful, branching agent workflows. The
flybrain topology gives you a **biologically-grounded** state graph:

- Each node = a neural role (sensory, motor, associative, etc.)
- Each edge = a real synapse class (excitatory, inhibitory, modulatory)
- Each module = a brain region (cerebrum, cerebellum, etc.)

This is not just a graph structure — it's a **provably-traceable**
agent topology that can be audited with `flybrain audit`.

## Next

- [05_export_crewai.ipynb](05_export_crewai.ipynb) — export to CrewAI format